# C++ Python Interface

This notebook introduces the pybind11 extension exposed as `conformer_toolkit_cpp`. It covers loading XYZ conformers, duplicate removal, atom index cleanup, writing cleaned records, and ring detection.

## Setup

Build the extension first with `pixi run build` from the repository root. The setup cell below locates the repository root and adds `src/` to `sys.path`.

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "testdata").is_dir():
            return candidate
    raise RuntimeError("Could not find repository root")

repo = find_repo_root(Path.cwd().resolve())
src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from conformer_toolkit_cpp import Conformer_Group

data = src / "testdata"
data

PosixPath('/home/ah494165/RCAML/conformer_toolkit/src/testdata')

## Load XYZ conformers

`Conformer_Group.from_xyz_files` loads one conformer from each XYZ path. Use `records()` to inspect source metadata and atom counts.

In [2]:
group = Conformer_Group.from_xyz_files([
    str(data / "h2_a.xyz"),
    str(data / "h2_translated.xyz"),
    str(data / "h2_stretched.xyz"),
])

print("conformer count:", len(group))
for record in group.records():
    print(record.input_index, Path(record.source).name, len(record.mol.symbols), record.mol.symbols)

conformer count: 3
0 h2_a.xyz 2 ['H', 'H']
1 h2_translated.xyz 2 ['H', 'H']
2 h2_stretched.xyz 2 ['H', 'H']


## Remove duplicates

`remove_duplicates` keeps the first representative of each unique conformer group. The H2 translated structure is a duplicate of the reference, while the stretched structure is distinct at this tolerance.

In [3]:
result = group.remove_duplicates(tolerance=1e-3, run_index_cleanup=False, charge=0)

print("unique:", len(result.unique))
for unique in result.unique:
    print(" ", unique.input_index, Path(unique.path).name)

print("duplicates:", len(result.duplicates))
for duplicate in result.duplicates:
    print(
        " ",
        Path(duplicate.path).name,
        "rep=",
        Path(duplicate.representative_path).name,
        "rmsd=",
        f"{duplicate.rmsd:.8f}",
    )

unique: 2
  0 h2_a.xyz
  2 h2_stretched.xyz
duplicates: 1
  h2_translated.xyz rep= h2_a.xyz rmsd= 0.00000000


## Normalize atom order

`index_cleanup` maps all conformers onto the atom order of the first conformer. This is useful when XYZ files contain the same molecule but a different atom ordering.

In [4]:
benzene = Conformer_Group.from_xyz_files([
    str(data / "benzene_ordered.xyz"),
    str(data / "benzene_reordered.xyz"),
])

before = [list(record.mol.symbols[:6]) for record in benzene.records()]
benzene.index_cleanup(max_mappings=1_000_000, bond_scale=1.3, ambiguity_gap=1e-6, charge=0)
after = [list(record.mol.symbols[:6]) for record in benzene.records()]

print("first six symbols before cleanup:", before)
print("first six symbols after cleanup:", after)

deduped = benzene.remove_duplicates(tolerance=1e-3, run_index_cleanup=False, charge=0)
print("unique after cleanup:", len(deduped.unique))

first six symbols before cleanup: [['C', 'C', 'C', 'C', 'C', 'C'], ['H', 'C', 'C', 'H', 'H', 'C']]
first six symbols after cleanup: [['C', 'C', 'C', 'C', 'C', 'C'], ['C', 'C', 'C', 'C', 'C', 'C']]
unique after cleanup: 1


## Write cleaned XYZ files

`write_records` writes the currently stored records. This example writes into `tutorials/output/`, which is ignored by git if you add it to your local excludes.

In [ ]:
outdir = repo / "tutorials" / "output" / "cleaned_benzene"
outdir.mkdir(parents=True, exist_ok=True)
benzene.write_records(str(outdir), "benzene")

sorted(path.name for path in outdir.glob("*.xyz"))

## Detect rings

`detect_rings` infers chemistry from the first conformer, stores the full molecule adjacency table for the group, and stores RDKit ring records. Ring atom adjacency includes the two ring neighbors plus directly bonded non-ring atoms.

In [5]:
ring_group = Conformer_Group.from_xyz_files([str(data / "benzene_ordered.xyz")])
ring_group.detect_rings(bond_scale=1.3, charge=0)

print("molecule adjacency:")
for atom, neighbors in enumerate(ring_group.adjacency_table()):
    print(" ", atom, "->", list(neighbors))

print("rings:")
for ring in ring_group.rings():
    print("ring atoms:", list(ring.atoms))
    for adjacency in ring.adjacency_list:
        print(" ", adjacency.atom, "->", list(adjacency.adjacent_atoms))

ring atoms: [0, 5, 4, 3, 2, 1]
  0 -> [1, 5]
  5 -> [0, 4]
  4 -> [5, 3]
  3 -> [4, 2]
  2 -> [3, 1]
  1 -> [2, 0]


## Parse metadata and select low-energy conformers

Pass a template matching the XYZ comment line to populate each record's `properties`. Property values are stored as strings; energy operations validate and convert the selected property to a finite number. A placeholder whose name starts with `_`, such as `{_framenumber}`, is used for matching but is not stored in `properties`. Each operation changes its group in place, so this example reloads the fixture for each independent selection.

In [ ]:
energy_path = data / "h2_energies.xyz"
comment_template = "frame = {frame} energy = {energy}"

def load_energy_group():
    return Conformer_Group.from_multi_xyz(
        str(energy_path), comment_template=comment_template
    )

energy_group = load_energy_group()
print([record.properties for record in energy_group.records()])

energy_only_group = Conformer_Group.from_multi_xyz(
    str(energy_path),
    comment_template="frame = {_framenumber} energy = {energy}",
)
print("placeholder omitted:", energy_only_group.records()[0].properties)

energy_group.sort_by_energy()
print("sorted frames:", [r.properties["frame"] for r in energy_group.records()])

lowest_half = load_energy_group()
lowest_half.retain_lowest_energy_percent(50.0)
print("lowest 50%:", [r.properties["frame"] for r in lowest_half.records()])

populated = load_energy_group()
populated.filter_by_boltzmann_population_ratio(
    minimum_ratio=0.5,
    temperature_kelvin=298.15,
    energy_to_joules_per_mole=1000.0,  # fixture energies are kJ/mol
)
print("population ratio >= 0.5:", [r.properties["frame"] for r in populated.records()])